# 🇮🇳 FastSpeech 2 — Telugu TTS Training Notebook

This notebook trains a **FastSpeech 2** Text-to-Speech model on the **Telugu** language using the IndicTTS dataset.
It covers dataset download & preprocessing, phonemization, forced alignment (MFA), model training, and inference with HiFi-GAN vocoder.
Results (generated audio + variance adaptor plots) are saved to Google Drive.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU!'}")
!nvidia-smi

## Step 1 — Install Dependencies

Install system-level TTS tools (`espeak-ng`, `sox`, `ffmpeg`), Python packages for phonemization and audio processing,
and the **Montreal Forced Aligner (MFA)** with Telugu acoustic model and dictionary.

In [ ]:
!apt-get install -y espeak-ng espeak-ng-data libespeak-ng-dev flite festival
!apt-get install -y sox ffmpeg

!pip install -q phonemizer epitran num2words tgt pyworld librosa soundfile pyyaml tqdm \
    indic-transliteration datasets huggingface_hub seaborn \
    indic-nlp-library

# Install MFA
!conda install -c conda-forge montreal-forced-aligner -y -q
# Download Telugu MFA models if available, else fallback
!mfa model download acoustic telugu_mfa 2>/dev/null || echo "Telugu MFA model not found — will use fallback alignment"
!mfa model download dictionary telugu_mfa 2>/dev/null || echo "Telugu MFA dictionary not found — will use fallback"

print("✅ All dependencies installed!")

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/FastSpeech2/{Telugu,model,utils,Results/Telugu/{test_generated_audios,variance_adaptor_plots/{duration,pitch,energy}}}
os.chdir('/content/FastSpeech2')
print("✅ Working directory:", os.getcwd())

## Step 2 — Upload Project Files

Upload the FastSpeech 2 project files (scripts, configs, model definitions) to `/content/FastSpeech2`.
You can either:
- **Option A**: Extract from a zip uploaded to Google Drive.
- **Option B**: Clone the repository from GitHub.

Ensure the following are present after this step:
- `Telugu/preprocess.py`
- `Telugu/train.py`
- `Telugu/inference.py`
- `Telugu/config.yaml`
- `model/` and `utils/` directories

In [ ]:
import os, zipfile, shutil

os.chdir('/content/FastSpeech2')

# --- Option A: Extract from Google Drive zip ---
drive_zip = '/content/drive/MyDrive/FastSpeech2/FastSpeech2_Telugu.zip'
if os.path.exists(drive_zip):
    print(f"📦 Found zip at {drive_zip}. Extracting...")
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall('/content/FastSpeech2')
    print("✅ Project files extracted from Drive zip.")

# --- Option B: Clone from GitHub (replace URL with your repo) ---
# elif not os.path.exists('/content/FastSpeech2/Telugu/train.py'):
#     !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/FastSpeech2
#     print("✅ Project files cloned from GitHub.")

else:
    print("⚠️ No zip found on Drive. If using GitHub, uncomment Option B above.")
    print("   Or manually upload files to /content/FastSpeech2 using the Files panel.")

# Verify key files
for f in ['Telugu/preprocess.py', 'Telugu/train.py', 'Telugu/inference.py', 'Telugu/config.yaml']:
    status = '✅' if os.path.exists(f'/content/FastSpeech2/{f}') else '❌ MISSING'
    print(f"  {status}  {f}")

## Step 3 — Download IndicTTS Telugu Dataset

We attempt to load the Telugu TTS dataset from **HuggingFace** (`ai4bharat/IndicTTS` or similar).
If unavailable, follow the manual download instructions printed below.

> **Manual source**: [IIT Madras IndicTTS](https://www.iitm.ac.in/donlab/tts/database.php)

In [ ]:
import os, numpy as np, soundfile as sf, librosa, csv
from tqdm import tqdm
os.chdir('/content/FastSpeech2')

print("Attempting to load IndicTTS Telugu from HuggingFace...")

try:
    from datasets import load_dataset
    # Try multiple possible dataset names
    for ds_name in ["ai4bharat/IndicTTS", "Telugu_TTS", "ai4bharat/indic-tts-te"]:
        try:
            dataset = load_dataset(ds_name, "te", trust_remote_code=True)
            print(f"✅ Loaded from: {ds_name}")
            break
        except Exception as e:
            print(f"  ✗ {ds_name}: {e}")
    else:
        raise Exception("None of the HuggingFace sources worked")

except Exception as e:
    print(f"HuggingFace not available: {e}")
    print("""
    📥 Manual Download Instructions:
    1. Go to: https://www.iitm.ac.in/donlab/tts/database.php
    2. Download Telugu TTS dataset (female or male speaker)
    3. Upload to Google Drive at: /content/drive/MyDrive/FastSpeech2/indic_tts_telugu.zip
    4. Run the next cell to extract it
    """)
    dataset = None

In [ ]:
import pandas as pd

os.makedirs('/content/FastSpeech2/data/raw/telugu/wavs', exist_ok=True)

TARGET_TRAIN_HOURS = 5.0
TARGET_VAL_HOURS = 1.0

if dataset is not None:
    # Process HuggingFace dataset
    split_data = dataset.get('train', dataset[list(dataset.keys())[0]])

    TARGET_TOTAL = (TARGET_TRAIN_HOURS + TARGET_VAL_HOURS) * 3600
    accumulated = 0
    selected = []

    for i, sample in enumerate(split_data):
        audio = sample['audio']
        duration = len(audio['array']) / audio['sampling_rate']
        if accumulated < TARGET_TOTAL:
            selected.append((i, duration, sample))
            accumulated += duration
        if accumulated >= TARGET_TOTAL:
            break

    print(f"Selected {len(selected)} samples = {accumulated/3600:.2f} hours")

    # Split train/val
    train_cutoff = TARGET_TRAIN_HOURS * 3600
    metadata = []
    cum = 0

    for i, (idx, dur, sample) in enumerate(tqdm(selected, desc="Saving Telugu audio")):
        split = "train" if cum < train_cutoff else "val"
        audio_array = np.array(sample['audio']['array'], dtype=np.float32)
        orig_sr = sample['audio']['sampling_rate']
        text = sample.get('text', sample.get('sentence', sample.get('transcription', '')))

        if orig_sr != 22050:
            audio_array = librosa.resample(audio_array, orig_sr=orig_sr, target_sr=22050)

        file_id = f"telugu_{idx:06d}"
        wav_path = f"/content/FastSpeech2/data/raw/telugu/wavs/{file_id}.wav"
        sf.write(wav_path, audio_array, 22050)
        metadata.append({'file_id': file_id, 'text': text, 'duration': dur, 'split': split})
        cum += dur

else:
    # Process manually uploaded dataset
    zip_path = '/content/drive/MyDrive/FastSpeech2/indic_tts_telugu.zip'
    if os.path.exists(zip_path):
        import zipfile, shutil
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/indic_tts_raw')

        # Find wav files and transcription
        wav_files = []
        for root, dirs, files in os.walk('/content/indic_tts_raw'):
            for f in files:
                if f.endswith('.wav'):
                    wav_files.append(os.path.join(root, f))

        # Sort and select ~6hrs
        metadata = []
        cum = 0
        TARGET_TOTAL = (TARGET_TRAIN_HOURS + TARGET_VAL_HOURS) * 3600

        for wav_path in tqdm(sorted(wav_files), desc="Processing Telugu wavs"):
            audio, sr = librosa.load(wav_path, sr=22050)
            duration = len(audio) / 22050
            if cum >= TARGET_TOTAL:
                break

            split = "train" if cum < TARGET_TRAIN_HOURS * 3600 else "val"

            # Try to find transcript
            txt_path = wav_path.replace('.wav', '.txt')
            text = open(txt_path, encoding='utf-8').read().strip() if os.path.exists(txt_path) else ""

            file_id = os.path.splitext(os.path.basename(wav_path))[0]
            dst = f"/content/FastSpeech2/data/raw/telugu/wavs/{file_id}.wav"
            sf.write(dst, audio, 22050)
            metadata.append({'file_id': file_id, 'text': text, 'duration': duration, 'split': split})
            cum += duration
    else:
        print("⚠️ No dataset found! Please upload indic_tts_telugu.zip to Google Drive.")
        metadata = []

# Save metadata
with open('/content/FastSpeech2/data/raw/telugu/metadata.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['file_id', 'text', 'duration', 'split'])
    writer.writeheader()
    writer.writerows(metadata)
print(f"✅ Saved {len(metadata)} Telugu audio files")

## Step 4 — Telugu Preprocessing (Phonemization + MFA)

Telugu uses **aksharas** (syllabic characters) as the fundamental phonetic unit.
We use **Epitran** to convert Telugu script to IPA, then run **MFA** (Montreal Forced Aligner)
to generate phoneme-level duration alignments (`.TextGrid` files).

If the Telugu MFA model is unavailable, the preprocessing script falls back to
character-level duration estimation.

In [ ]:
# Telugu uses character-level phonemes (aksharas)
# We use epitran for IPA conversion

import sys
sys.path.insert(0, '/content/FastSpeech2')

os.chdir('/content/FastSpeech2')
!python Telugu/preprocess.py --config Telugu/config.yaml
print("✅ Telugu preprocessing complete!")

## Step 5 — Training

Train the FastSpeech 2 model on the preprocessed Telugu data.
Training logs are streamed to the notebook and also saved to Google Drive.

> **Tip**: On a Colab T4 GPU, expect ~8–12 hours for a full run on 5 hours of data.
> Use `Colab Pro` or resume from a checkpoint if the session disconnects.

In [ ]:
os.chdir('/content/FastSpeech2')
!python Telugu/train.py --config Telugu/config.yaml 2>&1 | tee /content/drive/MyDrive/FastSpeech2/telugu_training.log
print("✅ Telugu training complete!")

## Step 6 — Inference on Telugu Test Sentences

Run inference using the best checkpoint and **HiFi-GAN** vocoder.
The following Telugu test sentences are used (defined in `Telugu/config.yaml`):

| ID  | Telugu Sentence |
|-----|-----------------|
| T1  | నమస్కారం, మీరు ఎలా ఉన్నారు? |
| T2a | నేను తెలుగు మాట్లాడతాను. |
| T2b | ఇది ఒక అందమైన రోజు. |
| T3  | భారతదేశం గొప్ప దేశం. |
| T4  | సంగీతం ఆత్మకు ఆహారం. |

In [ ]:
os.chdir('/content/FastSpeech2')

# Setup HiFi-GAN if not already done
if not os.path.exists('/content/hifi-gan'):
    !git clone https://github.com/jik876/hifi-gan /content/hifi-gan
    os.chdir('/content/hifi-gan')
    !pip install -q -r requirements.txt
    os.chdir('/content/FastSpeech2')

!python Telugu/inference.py \
  --checkpoint Telugu/checkpoints/best_checkpoint.pth \
  --config Telugu/config.yaml \
  --output_dir Results/Telugu \
  --vocoder_path /content/hifi-gan

print("✅ Telugu inference complete!")

In [ ]:
import IPython.display as ipd
import glob, matplotlib.pyplot as plt
from PIL import Image

print("=== Generated Telugu Audio ===")
for wav_file in sorted(glob.glob('/content/FastSpeech2/Results/Telugu/test_generated_audios/*.wav')):
    print(f"\n{os.path.basename(wav_file)}")
    display(ipd.Audio(wav_file))

print("\n=== Variance Adaptor Plots ===")
fig, axes = plt.subplots(3, 5, figsize=(20, 10))
for col, sent_id in enumerate(['T1', 'T2a', 'T2b', 'T3', 'T4']):
    for row, ptype in enumerate(['duration', 'pitch', 'energy']):
        path = f'/content/FastSpeech2/Results/Telugu/variance_adaptor_plots/{ptype}/{sent_id}_{ptype}.png'
        if os.path.exists(path):
            img = Image.open(path)
            axes[row][col].imshow(img)
            axes[row][col].axis('off')
            if row == 0:
                axes[row][col].set_title(sent_id, fontsize=12)
            if col == 0:
                axes[row][col].set_ylabel(ptype, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
import shutil
drive_output = '/content/drive/MyDrive/FastSpeech2/Results/Telugu'
os.makedirs(drive_output, exist_ok=True)
shutil.copytree('/content/FastSpeech2/Results/Telugu', drive_output, dirs_exist_ok=True)

drive_ckpt = '/content/drive/MyDrive/FastSpeech2/Telugu/checkpoints'
os.makedirs(drive_ckpt, exist_ok=True)
shutil.copytree('/content/FastSpeech2/Telugu/checkpoints', drive_ckpt, dirs_exist_ok=True)
print("✅ Telugu results saved to Google Drive!")